# BioEmu2 on Colab

Runs Microsoft Research's [BioEmu2](https://github.com/microsoft/bioemu2) (public preview, MIT licence) in three modes:

1. **Structure prediction**: one sharp, most-likely structure.
2. **Equilibrium sampling**: an ensemble of independent conformations.
3. **Time propagation**: chains of time-lagged endpoints (1 ns – 1 µs per step), a coarse trajectory.

The code is adapted from the examples in the BioEmu2 repository and turned into a Colab notebook, so that the model can be tried without a local GPU or a local installation. BioEmu2 is installed from that repository and its weights are downloaded from Hugging Face when you run the cells; neither is included here. Portions adapted from microsoft/bioemu2: Copyright (c) Microsoft Corporation, licensed under the MIT License.

**Before you start:** *Runtime → Change runtime type → GPU*. A free **T4** works for small proteins; with Colab Pro an **L4** (about 2 compute units/hour) is a good default, and an **A100** is faster for large complexes.

**Typical times on an L4** (200 denoising steps per sample): ~200 residues ≈ 9 s/sample at batch 10; ~400 residues ≈ 20 s/sample at batch 5. Install ≈ 3 min, model download ≈ 20 s.

**Notes**
- BioEmu2 is research software. Sampling frequencies are *not* calibrated Boltzmann probabilities, and there is no confidence score (no pLDDT/PAE).
- MSAs come from the public ColabFold server unless you upload your own A3M file.
- This notebook installs BioEmu2 from GitHub, pinned to commit `6d1ca74` (3 Nov 2026), with the CUDA 12 build, which matches Colab's GPU drivers.

## 1. Check the GPU

In [ ]:
#@title Check GPU and Python version
import subprocess, sys
print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 12), "BioEmu2 needs Python 3.12 or newer."
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv"],
                         capture_output=True, text=True, check=True).stdout)
except Exception:
    raise SystemExit("No GPU found. Use Runtime → Change runtime type → GPU, then run this cell again.")

## 2. Install BioEmu2 (takes a few minutes)

In [ ]:
#@title Install
!pip install -q "bioemu2[cuda12] @ git+https://github.com/microsoft/bioemu2@6d1ca744fa8ed285edafae7b98da65205bf00fd3" py3Dmol \
    --extra-index-url https://download.pytorch.org/whl/cu126 2>&1 | tail -n 20

In [ ]:
#@title Check the install
import importlib
try:
    import torch, bioemu2
    print("bioemu2 imported; torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except Exception as err:
    print("Import failed:", err)
    print("If torch was already loaded, use Runtime → Restart session, then continue from this cell.")

## 3. Define the system
Give either a **PDB ID** (chains and ligands from the first biological assembly) or **sequence(s)**.
For a complex from sequences, separate chains with `:`. A ligand can be added by its CCD code (e.g. `ATP`) or a SMILES string.
Tip: to drop expression tags or termini that are disordered in the crystal, paste the trimmed chain sequences with `input_type = sequence` instead of using the PDB ID.

In [ ]:
#@title Input
input_type = "pdb_id"  #@param ["pdb_id", "sequence"]
pdb_id = "1AKE"  #@param {type:"string"}
include_pdb_ligands = False  #@param {type:"boolean"}
sequence = ""  #@param {type:"string"}
ligand_ccd_code = ""  #@param {type:"string"}
ligand_smiles = ""  #@param {type:"string"}
use_own_msa = False  #@param {type:"boolean"}
job_name = "bioemu2_run"  #@param {type:"string"}

import string
from pathlib import Path
from bioemu2 import System, Protein, Ligand, get_msa

out_root = Path("/content") / job_name
out_root.mkdir(parents=True, exist_ok=True)

if input_type == "pdb_id":
    system = System.from_pdb_id(pdb_id.strip(), include_ligands=include_pdb_ligands)
    entities = list(system.entities)
else:
    chains = [s.strip().upper().replace(" ", "") for s in sequence.split(":") if s.strip()]
    assert chains, "Enter at least one protein sequence."
    ids = iter(string.ascii_uppercase)
    entities = [Protein(sequence=s, asym_ids=[next(ids)]) for s in chains]
    if ligand_ccd_code.strip():
        entities.append(Ligand(ccd_code=ligand_ccd_code.strip().upper(), asym_ids=[next(ids)]))
    if ligand_smiles.strip():
        entities.append(Ligand(smiles=ligand_smiles.strip(), asym_ids=[next(ids)]))
    system = System(entities=entities)

print(system)

if use_own_msa:
    from google.colab import files
    assert sum(isinstance(e, Protein) for e in entities) == 1, "Own-MSA upload here supports one protein chain only."
    uploaded = files.upload()
    a3m_path = Path("/content") / next(iter(uploaded))
    msa = get_msa(system, source=a3m_path)
else:
    msa = get_msa(system)  # ColabFold server, cached after the first call
print("MSA ready.")

## 4. Load the model

In [ ]:
#@title Load BioEmu2 weights (downloaded from Hugging Face on first use)
import torch
from bioemu2 import load_model

# bfloat16 needs an Ampere-or-newer GPU (A100, L4, …); a T4 falls back to full float32.
major, _ = torch.cuda.get_device_capability()
precision = "bf16-mixed" if major >= 8 else "32-true"
model = load_model(precision=precision)
print("Model loaded with precision", precision, "on", torch.cuda.get_device_name())

## 5. Structure prediction

In [ ]:
#@title Predict the structure
num_structures = 1  #@param {type:"integer"}
seed = 0  #@param {type:"integer"}

from bioemu2 import EmuSession, save_frames, check_physicality
from bioemu2.visualization import view_structure

emu_structure = EmuSession(system, model=model, mode="structure", msa=msa)
prediction = emu_structure.sample(num_samples=num_structures, seed=seed)
save_frames(prediction, out_root / "structure")
print("Saved to", out_root / "structure")
view_structure(prediction, frame=0).show()

## 6. Equilibrium sampling
Independent samples from the learned ensemble. On an L4, 50 samples of a ~200-residue protein take about 7–8 min. Lower `batch_size` if you run out of memory (e.g. 5 for ~400 residues).

In [ ]:
#@title Sample the ensemble
num_samples = 50  #@param {type:"integer"}
batch_size = 10  #@param {type:"integer"}
seed = 1  #@param {type:"integer"}
physicality_mode = "high_quality"  #@param ["high_quality", "high_quality_per_residue", "no_violations"]

emu_eq = EmuSession(system, model=model, mode="equilibrium", msa=msa)
samples = emu_eq.sample(num_samples=num_samples, batch_size=batch_size, seed=seed)

report = check_physicality(samples, physicality_mode, system=system)
save_frames(samples, out_root / "equilibrium", physicality=report)
n_ok = int(report.passed.sum())
print(f"{n_ok}/{samples.n_frames} samples pass the physicality check" + ("" if report.checked else " (not checked: no protein atoms)"))
print("Saved to", out_root / "equilibrium")

In [ ]:
#@title Plot RMSD to the predicted structure and radius of gyration
import numpy as np, mdtraj as md, matplotlib.pyplot as plt

ca = samples.topology.select("name CA")
rmsd_A = 10 * md.rmsd(samples, prediction, frame=0, atom_indices=ca)
rg_A = 10 * md.compute_rg(samples.atom_slice(ca))
ok = report.passed.astype(bool)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(rmsd_A[ok], bins=30, color="#3b6fb6"); ax[0].set_xlabel("Cα RMSD to predicted structure (Å)"); ax[0].set_ylabel("samples")
ax[1].hist(rg_A[ok], bins=30, color="#3b6fb6"); ax[1].set_xlabel("Cα radius of gyration (Å)")
ax[2].scatter(rmsd_A[ok], rg_A[ok], s=12, color="#3b6fb6", label="pass")
ax[2].scatter(rmsd_A[~ok], rg_A[~ok], s=12, color="#c2410c", label="fail physicality")
ax[2].set_xlabel("Cα RMSD (Å)"); ax[2].set_ylabel("Cα Rg (Å)"); ax[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
#@title View the ensemble (first frames, physically valid samples only)
from bioemu2.visualization import view_trajectory
max_frames = 20  #@param {type:"integer"}
view_trajectory(samples[np.where(ok)[0]] if ok.any() else samples, max_frames=max_frames).show()

## 7. Time propagation
Starts from some equilibrium samples and repeatedly propagates each by `lag_time_ns`. On an L4, one step for 4 starts of a ~200-residue protein takes about 35 s.
Each step is an independent endpoint at that time lag, not an MD frame in between. The documented lag range is 1–1000 ns.

In [ ]:
#@title Simulate time-lagged trajectories
num_starts = 4  #@param {type:"integer"}
lag_time_ns = 10.0  #@param {type:"number"}
num_steps = 10  #@param {type:"integer"}
seed = 2  #@param {type:"integer"}

start_idx = np.where(ok)[0][:num_starts] if ok.any() else np.arange(min(num_starts, samples.n_frames))
x = samples[start_idx]
steps = [x]
for i in range(num_steps):
    x = emu_eq.propagate(x, lag_time_ns=lag_time_ns, seed=seed + i, progress=False)
    steps.append(x)
    print(f"step {i + 1}/{num_steps} done ({(i + 1) * lag_time_ns:g} ns)", flush=True)

trajs = [md.join([s[j] for s in steps]) for j in range(len(start_idx))]
for j, t in enumerate(trajs):
    save_frames(t, out_root / "propagation" / f"start_{j}")
print("Saved to", out_root / "propagation")

time_ns = np.arange(num_steps + 1) * lag_time_ns
plt.figure(figsize=(7, 4))
for j, t in enumerate(trajs):
    plt.plot(time_ns, 10 * md.rmsd(t, prediction, frame=0, atom_indices=ca), marker="o", ms=3, label=f"start {j}")
plt.xlabel("time (ns)"); plt.ylabel("Cα RMSD to predicted structure (Å)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
#@title View one propagated trajectory
trajectory_to_view = 0  #@param {type:"integer"}
view_trajectory(trajs[trajectory_to_view], max_frames=num_steps + 1).show()

## 8. Test run: HPr kinase-HPr (1KKL)

A complex BioEmu2 handles reasonably well, to see what a working run looks like and to check the notebook end to end. It is the *Lactobacillus casei* HPr kinase with *Bacillus subtilis* HPr ([1KKL](https://www.rcsb.org/structure/1KKL)), given as three copies of the kinase plus one HPr: the kinase works as an oligomer, and the copies are what make the HPr site. The three kinase sequences differ by a residue or two at the termini because they are the constructs resolved in the crystal.

This cell defines its own system and leaves the one from step 3 alone. It reports how *consistent* the interface is across samples, which is the thing to look at: an ensemble that keeps docking the chains in a different place each time tells you the model has not determined the interface, even though every sample has plenty of contact.

The second cell puts that ensemble next to molecular dynamics of the same complex, taken from **DynaBench**: three 100 ns replicas for each of the 200-odd complexes of Docking Benchmark 5.5. The notebook carries 21 frames of the 1KKL trajectories, seven from each replica with the first 20 ns of each discarded; the full set is at [http://www-lbt.ibpc.fr/DynaBench](http://www-lbt.ibpc.fr/DynaBench) on the MDposit platform (INRIA node) of the Molecular Dynamics Data Bank. If you use them, cite Barlas AB, Laurent B, Karaca E, Prévost C, Sacquin-Mora S. DynaBench: Dynamic data for the docking benchmark. *Journal of Molecular Biology* 438 (2026) 169650, [doi:10.1016/j.jmb.2026.169650](https://doi.org/10.1016/j.jmb.2026.169650).


In [ ]:
#@title Run the 1KKL test { display-mode: "form" }
run_the_test = False #@param {type:"boolean"}
test_num_samples = 10 #@param {type:"integer"}
test_seed = 1 #@param {type:"integer"}
#@markdown 614 residues, so this is slower than the default example. Ten samples take a
#@markdown few minutes on an L4 and give coarse frequencies; the reference numbers printed
#@markdown underneath come from a 50-sample run.
import string

import numpy as np

TEST_CHAINS = [
    "ERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNEE",
    "ERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNE",
    "AERRSMHGVLVDIYGLGVLITGDSGVGKSETALELVQRGHRLIADDRVDVYQQDEQTIVGAAPPILSHLLEIRGLGIIDVMNLFGAGAVREDTTISLIVHLENWTPDKTFDRLGSGEQTQLIFDVPVPKITVPVKVGRNLAIIIEVAAMNFRAKSMGYDATKTFEKNLNHLIEHNEE",
    "QKTFKVTADSGIHARPATVLVQTASKYDADVNLEYNGKTVNLKSIMGVMSLGIAKGAEITISASGADENDALNALEETMKSERLGE"
]

if not run_the_test:
    print("Tick the box to run the test. It needs the model from step 4.")
elif "model" not in dir():
    print("Run step 4 first: the test uses the model loaded there.")
else:
    from bioemu2 import EmuSession, Protein, System, check_physicality, get_msa

    ids = iter(string.ascii_uppercase)
    test_system = System(entities=[Protein(sequence=s, asym_ids=[next(ids)]) for s in TEST_CHAINS])
    length = sum(map(len, TEST_CHAINS))
    print(f"{len(TEST_CHAINS)} chains, {length} residues. Building the MSA ...")
    test_msa = get_msa(test_system)
    test_emu = EmuSession(test_system, model=model, mode="equilibrium", msa=test_msa)
    test_samples = test_emu.sample(num_samples=test_num_samples,
                                   batch_size=max(1, 2000 // length), seed=test_seed)
    test_report = check_physicality(test_samples, "high_quality", system=test_system)
    print(f"{int(test_report.passed.sum())}/{test_samples.n_frames} samples pass the physicality check")

    # the interface between the third kinase copy (chain C) and HPr (chain D)
    top = test_samples.topology
    chains = list(top.chains)
    sel = [top.select(f"chainid {chains[i].index} and element != H") for i in (2, 3)]
    onehot = []
    for atoms in sel:
        res = np.array([top.atom(i).residue.index for i in atoms])
        res -= res.min()
        hot = np.zeros((len(atoms), res.max() + 1), dtype=np.float32)
        hot[np.arange(len(atoms)), res] = 1.0
        onehot.append(hot)

    freq = np.zeros((onehot[0].shape[1], onehot[1].shape[1]))
    touching = 0
    for f in range(test_samples.n_frames):
        xa = test_samples.xyz[f][sel[0]] * 10.0                      # A
        xb = test_samples.xyz[f][sel[1]] * 10.0
        close = (np.linalg.norm(xa[:, None, :] - xb[None, :, :], axis=-1) <= 4.5).astype(np.float32)
        pairs = (onehot[0].T @ close @ onehot[1]) > 0
        freq += pairs
        touching += int(pairs.any())
    freq /= test_samples.n_frames

    print(f"\nkinase chain C against HPr, over {test_samples.n_frames} samples")
    print(f"  samples with any contact          : {touching}/{test_samples.n_frames}")
    print(f"  distinct residue pairs touched    : {int((freq > 0).sum())}")
    print(f"  highest frequency of any one pair : {freq.max():.2f}")
    print(f"  pairs in at least half the samples: {int((freq >= 0.5).sum())}")
    print("\n  for comparison, a 50-sample run of this system gave:")
    print("  40/50 samples in contact, 257 distinct pairs, highest frequency 0.52, 2 pairs in half,")
    print("  and 7 samples within fnat > 0.3 of the crystal structure (best complex RMSD 2.9 A).")
    print("\n  the same measurement on a complex BioEmu2 does not resolve (PKR kinase-eIF2alpha,")
    print("  2A1A, submitted without the PKR dimer) gave 49/50 samples in contact but 1744 distinct")
    print("  pairs, highest frequency 0.12 and none in half: contact everywhere, interface nowhere.")


In [ ]:
#@title Look at the test ensemble next to an MD reference { display-mode: "form" }
max_frames_shown = 8 #@param {type:"integer"}
#@markdown The left viewer is the BioEmu2 ensemble you just sampled, the right one a
#@markdown molecular-dynamics reference for the same complex: 21 frames of the 1KKL
#@markdown trajectories of the [DynaBench](https://doi.org/10.1016/j.jmb.2026.169650) dataset,
#@markdown seven from each 100 ns replica with the first 20 ns of each discarded. Each ensemble is
#@markdown superposed on its own first frame, so what you see is how far each one wanders.
#@markdown Only the backbone is drawn, to keep the viewer light.
import urllib.request
from pathlib import Path

import mdtraj as md
import py3Dmol

MD_BASE = "https://raw.githubusercontent.com/CSB-KaracaLab/bioemu2-colab/main/md_reference/"
BACKBONE = "name N or name CA or name C or name O"


def _show_ensemble(view, traj, cell, colour, limit):
    """Draw the first few frames of a trajectory as translucent backbone ribbons."""
    thin = traj.atom_slice(traj.topology.select(BACKBONE))
    for i in range(min(limit, thin.n_frames)):
        thin[i].save_pdb("/tmp/_frame.pdb")
        view.addModel(Path("/tmp/_frame.pdb").read_text(), "pdb", viewer=cell)
    view.setStyle({}, {"cartoon": {"color": colour, "opacity": 0.4}}, viewer=cell)
    view.zoomTo(viewer=cell)


if "test_samples" not in dir():
    print("Run the 1KKL test above first.")
else:
    for name in ("1KKL_md_topology.pdb", "1KKL_md_dynabench.xtc"):
        if not Path(name).is_file():
            urllib.request.urlretrieve(MD_BASE + name, name)
    md_ref = md.load("1KKL_md_dynabench.xtc", top="1KKL_md_topology.pdb")
    md_ref.superpose(md_ref, 0, atom_indices=md_ref.topology.select("name CA"))

    ok = np.where(test_report.passed)[0] if test_report.passed.any() else np.arange(test_samples.n_frames)
    emu = test_samples[ok]
    emu.superpose(emu, 0, atom_indices=emu.topology.select("name CA"))

    spread = lambda t: 10 * md.rmsd(t, t, 0, atom_indices=t.topology.select("name CA")).max()
    print(f"BioEmu2 : {emu.n_frames} physical samples, Ca RMSD to its own first frame up to "
          f"{spread(emu):.1f} A")
    print(f"MD      : {md_ref.n_frames} frames, same measure up to {spread(md_ref):.1f} A")
    print(f"both are the same four chains: {[c.n_residues for c in md_ref.topology.chains]}")

    view = py3Dmol.view(width=900, height=420, viewergrid=(1, 2), linked=False)
    _show_ensemble(view, emu, (0, 0), "cornflowerblue", max_frames_shown)
    _show_ensemble(view, md_ref, (0, 1), "darkseagreen", max_frames_shown)
    view.show()
    print("\nleft: BioEmu2 equilibrium samples | right: the MD reference")


## 9. Download results

In [ ]:
#@title Zip and download everything
import shutil
from google.colab import files
archive = shutil.make_archive(str(out_root), "zip", out_root)
files.download(archive)

---
Each output folder has `topology.cif` + `samples.xtc` (open in PyMOL, ChimeraX or VMD, or load with MDTraj). `equilibrium/physicality.json` lists which samples failed the checks and why.

Sources: [microsoft/bioemu2](https://github.com/microsoft/bioemu2) README and docs; [model card](https://huggingface.co/microsoft/bioemu2).

The MD reference in the test run comes from the DynaBench dataset: Barlas AB, Laurent B, Karaca E, Prévost C, Sacquin-Mora S. DynaBench: Dynamic data for the docking benchmark. *Journal of Molecular Biology* 438 (2026) 169650, [doi:10.1016/j.jmb.2026.169650](https://doi.org/10.1016/j.jmb.2026.169650). The trajectories are distributed at [http://www-lbt.ibpc.fr/DynaBench](http://www-lbt.ibpc.fr/DynaBench); please cite the paper if you use them. DynaBench is also the name of the Karaca lab's MD analysis pipeline, [CSB-KaracaLab/DynaBench](https://github.com/CSB-KaracaLab/DynaBench), which is a different thing.
